# Diabetic Retinopathy Stage Detection - Training Notebook

This notebook builds a diabetic retinopathy (DR) screening model from retinal fundus photographs. It grades each image on the five-stage scale (No DR, Mild, Moderate, Severe, Proliferative), detects whether DR is present, checks whether the photo is good enough to grade, and reports how certain it is.



## 0. Setup

Install the extra packages, import libraries, create output folders, check the GPU, fix the random seed and define every setting in one place.

### 0.1 Install packages
`timm` provides the pretrained CNN; `onnx` and `onnxruntime` are used for the deployment export.

In [ ]:
!pip install -q timm onnx onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 77.9 MB/s eta 0:00:00


### 0.2 Imports and output folders
All figures, tables and logs are written under `/kaggle/working/results` so they can be downloaded for the report.

In [ ]:
import os, json, time, random, copy, zipfile, inspect, warnings
from pathlib import Path
from dataclasses import dataclass, asdict
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import timm
from tqdm.auto import tqdm

warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

WORKING_DIR = Path('/kaggle/working')
WORKING_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORKING_DIR)
for folder in ['results/figures', 'results/tables', 'results/logs', 'models/export']:
    Path(folder).mkdir(parents=True, exist_ok=True)


def show_table(title, table):
    # print a table as plain text
    print(f'\n=== {title} ===')
    print(table.to_string())


def save_figure(path, dpi=200):

    plt.tight_layout()
    plt.savefig(path, dpi=dpi)
    print(f'figure saved: {path}')
    plt.show()


print('torch version:', torch.__version__)
print('timm version :', timm.__version__)

torch version: 2.10.0+cu128
timm version : 1.0.26


### 0.3 Check the GPU
Training needs a GPU accelerator (P100 or T4).

In [ ]:
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU: {gpu.name} | memory: {gpu.total_memory / 1e9:.1f} GB')
else:
    print('GPU: NONE - turn on the accelerator in the notebook settings')

GPU: Tesla T4 | memory: 15.6 GB


### 0.4 Fix the random seed
The same seed gives the same splits, sampling and initial weights on every run.

In [ ]:
RANDOM_SEED = 42

def seed_everything(seed=RANDOM_SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

seed_everything()
print('random seed:', RANDOM_SEED)

random seed: 42


### 0.5 Configuration
Every hyperparameter is stored in one `config` object. It is saved inside each checkpoint and with the final results, so every number can be traced back to the settings that produced it.

In [ ]:
@dataclass
class Config:
    # data
    image_size: int = 384              # input resolution in pixels
    num_classes: int = 5               # DR stages 0-4
    num_quality_classes: int = 2       # gradable / ungradable
    val_frac: float = 0.15
    test_frac: float = 0.15
    seed: int = 42

    # preprocessing switches
    use_graham: bool = True            # illumination correction / edge enhancement
    use_clahe: bool = True             # contrast enhancement

    # model
    backbone: str = 'tf_efficientnetv2_s.in21k_ft_in1k'
    pretrained: bool = True            # start from ImageNet weights
    dropout: float = 0.4

    # optimisation
    epochs: int = 25
    batch_size: int = 32
    head_lr: float = 1e-3              # learning rate for the new output layers
    backbone_lr: float = 1e-4          # learning rate for the pretrained layers
    weight_decay: float = 1e-5
    warmup_steps: int = 300
    grad_clip: float = 1.0
    amp: bool = True                   # mixed precision
    use_ema: bool = True               # moving average of weights
    ema_decay: float = 0.999
    freeze_first_epoch: bool = True    # train heads only in epoch 0
    quality_loss_weight: float = 0.3
    label_smoothing: float = 0.05

    # class balancing
    sampler_beta: float = 0.999

    # early stopping
    patience: int = 4
    min_delta: float = 0.002

    # uncertainty
    mc_dropout_samples: int = 20

    # triage thresholds
    ungradable_threshold: float = 0.50
    uncertainty_threshold: float = 0.45
    confidence_threshold: float = 0.55

    # paths and hardware
    checkpoint_dir: str = '/kaggle/working/checkpoints'
    cache_dir: str = '/kaggle/working/cache'
    device: str = 'cuda'
    num_workers: int = 2

    def to_dict(self):
        return asdict(self)


config = Config()
config.device = 'cuda' if torch.cuda.is_available() else 'cpu'
config.amp = config.amp and config.device == 'cuda'
config.num_workers = min(4, os.cpu_count() or 2)

show_table('Configuration', pd.Series(config.to_dict()).to_frame('value'))


=== Configuration ===
                                                   value
image_size                                           384
num_classes                                            5
num_quality_classes                                    2
val_frac                                            0.15
test_frac                                           0.15
seed                                                  42
use_graham                                          True
use_clahe                                           True
backbone               tf_efficientnetv2_s.in21k_ft_in1k
pretrained                                          True
dropout                                              0.4
epochs                                                25
batch_size                                            32
head_lr                                            0.001
backbone_lr                                       0.0001
weight_decay                                     0.00001
warmup_s